# Questão 3: Aprendizado por reforço no pouso lunar (LunarLander-v3)

Notebook com o código do trabalho de Aprendizado por Reforço (Gymnasium + Stable-Baselines3), feito para rodar no **Google Colab**.

**O que o enunciado pede e onde está neste notebook**

| Item | Onde |
|---|---|
| 3a) Ambiente (estados, ações e recompensas) e agente | Seção 1 (ambiente) e Seções 3 e 4 (agentes DQN e PPO) |
| 3b) Evolução do aprendizado ao longo dos episódios | Seções 5 e 7 (curvas de recompensa) e Seção 10 (GIF do treino) |
| 3b) Caminho final encontrado pelo agente | Seção 8 (GIFs do agente treinado pousando) |
| Variação própria: vento | Seção 9 |

**Tempo de execução (Colab):** DQN e PPO de 300 mil passos levam alguns minutos cada. A continuação do PPO até 1 milhão de passos leva cerca de 25 minutos. As seções de vento (Seção 9) levam mais tempo e são opcionais para quem só quer conferir o método.

## 1. O ambiente: LunarLander-v3

Uma nave precisa pousar entre duas bandeiras controlando os propulsores.

- **Estado** (8 valores): posição x e y, velocidade em x e y, ângulo, velocidade angular e dois indicadores de contato (perna esquerda e perna direita no chão).
- **Ações** (4, discretas): 0 = não fazer nada, 1 = propulsor lateral esquerdo, 2 = propulsor principal, 3 = propulsor lateral direito.
- **Recompensa** (soma ao longo do episódio):
  - positiva ao se aproximar da plataforma e ao reduzir a velocidade;
  - cerca de +10 por perna que encosta no chão;
  - **+100** ao pousar com sucesso;
  - negativa ao se afastar, ao inclinar, ao ir rápido demais e ao gastar combustível (propulsores);
  - **-100** ao bater.
- **Meta:** recompensa média acima de **200** indica pouso consistente.
- **Fim do episódio:** pouso, queda ou limite de passos.

## 2. Preparação (Colab)

In [ ]:
!apt-get install -y swig > /dev/null
!pip install -q "gymnasium[box2d]" stable-baselines3 imageio imageio-ffmpeg

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import imageio
import gymnasium as gym
from IPython.display import Image
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import EvalCallback

BASE = "/content/drive/MyDrive/rl_lunar"
for sub in ["logs", "models", "ckpt"]:
    os.makedirs(f"{BASE}/{sub}", exist_ok=True)

# Hiperparâmetros do PPO usados em todo o trabalho
PPO_PARAMS = dict(
    n_steps=1024, batch_size=64, n_epochs=4,
    gamma=0.999, gae_lambda=0.98, ent_coef=0.01, seed=42,
)

In [ ]:
# Funções auxiliares usadas nas seções seguintes
def criar_env(vento=False, render_mode=None, log=None):
    """Cria o LunarLander (com ou sem vento) já envolvido pelo Monitor."""
    kwargs = {}
    if vento:
        kwargs = dict(enable_wind=True, wind_power=15.0, turbulence_power=1.5)
    env = gym.make("LunarLander-v3", render_mode=render_mode, **kwargs)
    return Monitor(env, log)


def avaliar(modelo, vento=False, n=100):
    media, desvio = evaluate_policy(modelo, criar_env(vento), n_eval_episodes=n)
    return media, desvio


def gravar_gif(modelo, caminho, seed=7, vento=False, max_passos=600):
    """Grava um episódio do agente (política determinística) em GIF."""
    env = criar_env(vento, render_mode="rgb_array")
    obs, _ = env.reset(seed=seed)
    frames, total = [], 0.0
    for _ in range(max_passos):
        frames.append(env.render())
        acao, _ = modelo.predict(obs, deterministic=True)
        obs, r, term, trunc, _ = env.step(int(acao))
        total += r
        if term or trunc:
            frames.append(env.render())
            break
    env.close()
    imageio.mimsave(caminho, frames, fps=30, loop=0)
    print(f"Salvo em {caminho} | recompensa: {total:.1f} | quadros: {len(frames)}")


def recompensas_por_semente(modelo, vento=False, n=20):
    """Joga n episódios (sementes 0..n-1) e devolve a lista (semente, recompensa)."""
    saida = []
    for seed in range(n):
        env = criar_env(vento)
        obs, _ = env.reset(seed=seed)
        total, fim = 0.0, False
        while not fim:
            acao, _ = modelo.predict(obs, deterministic=True)
            obs, r, term, trunc, _ = env.step(int(acao))
            total += r
            fim = term or trunc
        env.close()
        saida.append((seed, total))
    return saida


def gravar_melhor_e_pior(modelo, nome, vento=False, n=20, max_passos=600):
    res = sorted(recompensas_por_semente(modelo, vento, n), key=lambda x: x[1])
    (s_pior, r_pior), (s_melhor, r_melhor) = res[0], res[-1]
    print(f"{nome} | pior: semente {s_pior} ({r_pior:.1f}) | melhor: semente {s_melhor} ({r_melhor:.1f})")
    gravar_gif(modelo, f"{BASE}/{nome}_melhor.gif", seed=s_melhor, vento=vento, max_passos=max_passos)
    gravar_gif(modelo, f"{BASE}/{nome}_pior.gif", seed=s_pior, vento=vento, max_passos=max_passos)


def carregar_log(caminho):
    """Lê o CSV do Monitor (1 linha de comentário no topo): r = recompensa, l = passos."""
    df = pd.read_csv(caminho, skiprows=1)
    df["episodio"] = range(1, len(df) + 1)
    df["passos_acum"] = df["l"].cumsum()
    return df


def suavizar(serie, janela=50):
    return serie.rolling(janela, min_periods=1).mean()

### Agente aleatório (ponto de partida)

Antes de treinar, vale ver o ambiente funcionando com ações sorteadas. Esse é o comportamento "sem aprendizado".

In [ ]:
env = gym.make("LunarLander-v3", render_mode="rgb_array")
obs, _ = env.reset(seed=42)
frames = []
for _ in range(300):
    frames.append(env.render())
    obs, r, term, trunc, _ = env.step(env.action_space.sample())
    if term or trunc:
        break
env.close()
imageio.mimsave(f"{BASE}/aleatorio.gif", frames, fps=30, loop=0)
Image(f"{BASE}/aleatorio.gif")

## 3. Agente 1: DQN (300 mil passos)

O DQN aprende o valor de cada ação em cada estado (Q-values), com memória de experiências (*replay buffer*), rede-alvo e exploração por epsilon.

In [ ]:
env_dqn = criar_env(log=f"{BASE}/logs/dqn_base")

modelo_dqn = DQN(
    "MlpPolicy", env_dqn,
    learning_rate=6.3e-4,
    buffer_size=50_000,
    learning_starts=0,
    batch_size=128,
    gamma=0.99,
    train_freq=4,
    gradient_steps=-1,
    target_update_interval=250,
    exploration_fraction=0.12,
    exploration_final_eps=0.1,
    policy_kwargs=dict(net_arch=[256, 256]),
    verbose=1,
    seed=42,
)
modelo_dqn.learn(total_timesteps=300_000)
modelo_dqn.save(f"{BASE}/models/dqn_base")

## 4. Agente 2: PPO (300 mil passos)

O PPO ajusta diretamente a política, com atualizações limitadas (*clipping*) para manter o aprendizado estável.

In [ ]:
env_ppo = criar_env(log=f"{BASE}/logs/ppo_base")

modelo_ppo = PPO("MlpPolicy", env_ppo, verbose=1, **PPO_PARAMS)
modelo_ppo.learn(total_timesteps=300_000)
modelo_ppo.save(f"{BASE}/models/ppo_base")

In [ ]:
# Avaliação com 100 episódios (política determinística)
for nome, modelo in [("DQN", modelo_dqn), ("PPO", modelo_ppo)]:
    media, desvio = avaliar(modelo)
    print(f"{nome} (300 mil passos): recompensa média = {media:.1f} ± {desvio:.1f}")

**Resultado registrado na execução original (semente 42, 100 episódios):** DQN = 69,3 ± 65,2 e PPO = 79,8 ± 106,7. Nos dois casos a média ainda ficou abaixo da meta de 200 e a variância foi alta.

## 5. Evolução do aprendizado: DQN e PPO nos primeiros 300 mil passos

O `Monitor` registrou a recompensa de cada episódio de treino. A curva abaixo é uma média móvel de 50 episódios.

In [ ]:
dqn = carregar_log(f"{BASE}/logs/dqn_base.monitor.csv")
ppo = carregar_log(f"{BASE}/logs/ppo_base.monitor.csv")
print("Episódios DQN:", len(dqn), "| Episódios PPO:", len(ppo))

plt.figure(figsize=(10, 5))
plt.plot(dqn["passos_acum"], suavizar(dqn["r"]), label="DQN", linewidth=2)
plt.plot(ppo["passos_acum"], suavizar(ppo["r"]), label="PPO", linewidth=2)
plt.axhline(200, color="gray", linestyle="--", label="Meta (200)")
plt.xlabel("Passos de treino")
plt.ylabel("Recompensa por episódio (média móvel de 50)")
plt.title("LunarLander: DQN vs. PPO (300 mil passos)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f"{BASE}/grafico_recompensa_300k.png", dpi=150)
plt.show()

## 6. Continuando o PPO até 1 milhão de passos

O PPO continua do ponto onde parou (`reset_num_timesteps=False`). A cada 25 mil passos, um ambiente separado avalia o agente em 20 episódios (`EvalCallback`).

In [ ]:
env_eval = criar_env()
callback = EvalCallback(
    env_eval,
    best_model_save_path=f"{BASE}/models/ppo_melhor",
    log_path=f"{BASE}/logs/ppo_eval",
    eval_freq=25_000,
    n_eval_episodes=20,
    deterministic=True,
    verbose=1,
)
modelo_ppo.learn(total_timesteps=700_000, reset_num_timesteps=False, callback=callback)
modelo_ppo.save(f"{BASE}/models/ppo_1M")

In [ ]:
modelo_melhor = PPO.load(f"{BASE}/models/ppo_melhor/best_model")
for nome, modelo in [("PPO 1M (final)", modelo_ppo), ("PPO melhor (callback)", modelo_melhor)]:
    media, desvio = avaliar(modelo)
    print(f"{nome}: {media:.1f} ± {desvio:.1f}")

**Resultado registrado na execução original (100 episódios):** PPO 1M final = **245,4 ± 52,0** (acima da meta de 200). O "melhor modelo" escolhido pelo callback chegou a 256,1 ± 31,0, mas essa escolha tem viés de seleção, então o resultado principal é o **modelo final**.

In [ ]:
ppo_1M = carregar_log(f"{BASE}/logs/ppo_base.monitor.csv")

plt.figure(figsize=(10, 5))
plt.plot(ppo_1M["passos_acum"], suavizar(ppo_1M["r"]), label="PPO", color="tab:orange", linewidth=2)
plt.plot(dqn["passos_acum"], suavizar(dqn["r"]), label="DQN (300k)", color="tab:blue", linewidth=2, alpha=0.7)
plt.axhline(200, color="gray", linestyle="--", label="Meta (200)")
plt.axvline(300_000, color="red", linestyle=":", label="Fim do treino base (300k)")
plt.xlabel("Passos de treino")
plt.ylabel("Recompensa por episódio (média móvel de 50)")
plt.title("LunarLander: PPO até 1 milhão de passos")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f"{BASE}/grafico_ppo_1M.png", dpi=150)
plt.show()

In [ ]:
dados = np.load(f"{BASE}/logs/ppo_eval/evaluations.npz")
passos = dados["timesteps"]
medias = dados["results"].mean(axis=1)
desvios = dados["results"].std(axis=1)

plt.figure(figsize=(10, 4))
plt.plot(passos, medias, marker="o", color="tab:orange")
plt.fill_between(passos, medias - desvios, medias + desvios, alpha=0.2, color="tab:orange")
plt.axhline(200, color="gray", linestyle="--")
plt.xlabel("Passos de treino")
plt.ylabel("Recompensa (avaliação, 20 episódios)")
plt.title("PPO: avaliação a cada 25 mil passos")
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f"{BASE}/avaliacao_ppo_1M.png", dpi=150)
plt.show()

## 7. Resumo: antes e depois do treino longo

In [ ]:
nomes = ["PPO 300k\n(antes)", "PPO 1M\n(final)", "PPO 1M\n(melhor)"]
medias = [79.8, 245.4, 256.1]      # valores registrados na execução original
desvios = [106.7, 52.0, 31.0]
cores = ["tab:gray", "tab:orange", "tab:green"]

plt.figure(figsize=(7, 5))
barras = plt.bar(nomes, medias, yerr=desvios, capsize=6, color=cores, alpha=0.85)
plt.axhline(200, color="gray", linestyle="--", label="Meta (200)")
for b, m in zip(barras, medias):
    plt.text(b.get_x() + b.get_width() / 2, 5, f"{m:.1f}", ha="center", fontweight="bold")
plt.ylabel("Recompensa média (100 episódios)")
plt.title("LunarLander (PPO): antes vs. depois do treino longo")
plt.legend(); plt.grid(axis="y", alpha=0.3); plt.tight_layout()
plt.savefig(f"{BASE}/resumo_antes_depois.png", dpi=150)
plt.show()

## 8. O caminho final: GIFs do agente pousando (e errando)

Cada GIF mostra um episódio completo do agente treinado. Gravamos também os **episódios de falha**, para mostrar onde o agente erra.

In [ ]:
modelo_dqn = DQN.load(f"{BASE}/models/dqn_base")
modelo_ppo_300k = PPO.load(f"{BASE}/models/ppo_base")   # atenção: 'ppo_base' é o PPO de 300k
modelo_ppo_1M = PPO.load(f"{BASE}/models/ppo_1M")

gravar_gif(modelo_ppo_300k, f"{BASE}/ppo_300k.gif", seed=7)   # antes
gravar_gif(modelo_ppo_1M, f"{BASE}/ppo_1M.gif", seed=7)       # depois
Image(f"{BASE}/ppo_1M.gif")

In [ ]:
# Melhor e pior episódio entre 20 sementes
gravar_melhor_e_pior(modelo_dqn, "dqn_300k")
gravar_melhor_e_pior(modelo_ppo_1M, "ppo_1M")
Image(f"{BASE}/dqn_300k_pior.gif")

## 9. Variação própria: vento

O ambiente aceita vento (`enable_wind=True`). Usamos `wind_power=15` e `turbulence_power=1.5`. As perguntas:

1. O agente treinado **sem vento** aguenta o vento?
2. Treinar **com vento** resolve?

In [ ]:
# Generalização: o agente treinado sem vento (PPO 1M), testado com e sem vento
m_sem, d_sem = avaliar(modelo_ppo_1M, vento=False)
m_com, d_com = avaliar(modelo_ppo_1M, vento=True)
print(f"Treinado SEM vento | testado sem vento: {m_sem:.1f} ± {d_sem:.1f}")
print(f"Treinado SEM vento | testado com vento: {m_com:.1f} ± {d_com:.1f}")

gravar_gif(modelo_ppo_1M, f"{BASE}/ppo_sem_vento_testado_com_vento.gif", seed=7, vento=True, max_passos=1000)

**Resultado registrado (100 episódios):** treinado sem vento, o agente faz **253,4 ± 39,2** sem vento e cai para **103,4 ± 131,9** com vento. A variância explode.

In [ ]:
# Treino COM vento (PPO, 1 milhão de passos)
env_treino_vento = criar_env(vento=True, log=f"{BASE}/logs/ppo_vento")
callback_vento = EvalCallback(
    criar_env(vento=True),
    best_model_save_path=f"{BASE}/models/ppo_vento_melhor",
    log_path=f"{BASE}/logs/ppo_vento_eval",
    eval_freq=25_000,
    n_eval_episodes=20,
    deterministic=True,
    verbose=1,
)

modelo_vento = PPO("MlpPolicy", env_treino_vento, verbose=1, **PPO_PARAMS)
modelo_vento.learn(total_timesteps=1_000_000, callback=callback_vento)
modelo_vento.save(f"{BASE}/models/ppo_vento")

modelo_vento_melhor = PPO.load(f"{BASE}/models/ppo_vento_melhor/best_model")
for nome, modelo in [("PPO vento (final)", modelo_vento), ("PPO vento (melhor)", modelo_vento_melhor)]:
    for vento in (False, True):
        m, d = avaliar(modelo, vento=vento)
        print(f"{nome} | testado {'com' if vento else 'sem'} vento: {m:.1f} ± {d:.1f}")

**Resultado registrado (100 episódios):** treinado com vento (1 milhão de passos), o agente faz 118,5 ± 74,2 sem vento e **93,3 ± 100,7** com vento. Treinar com vento **não ajudou** a superar o agente treinado sem vento.

In [ ]:
# (Opcional, ~1 hora) Mais 1 milhão de passos com vento
env_treino_vento2 = criar_env(vento=True, log=f"{BASE}/logs/ppo_vento_2M")
modelo_vento2 = PPO.load(f"{BASE}/models/ppo_vento", env=env_treino_vento2)
callback_vento2 = EvalCallback(
    criar_env(vento=True),
    best_model_save_path=f"{BASE}/models/ppo_vento_2M_melhor",
    log_path=f"{BASE}/logs/ppo_vento_2M_eval",
    eval_freq=25_000,
    n_eval_episodes=50,
    deterministic=True,
    verbose=1,
)
modelo_vento2.learn(total_timesteps=1_000_000, reset_num_timesteps=False, callback=callback_vento2)
modelo_vento2.save(f"{BASE}/models/ppo_vento_2M")

m, d = avaliar(modelo_vento2, vento=True)
print(f"Vento 2M (final), testado com vento: {m:.1f} ± {d:.1f}")

**Resultado registrado:** com 2 milhões de passos o agente terminou instável (**27,3 ± 156,9** com vento), sem superar o agente treinado sem vento.

**Limitação:** todos os treinos usaram uma única semente (42). Não dá para afirmar, só com esses dados, que o vento é intransponível para o PPO.

## 10. O agente aprendendo: o mesmo cenário em 5 momentos do treino

Para mostrar a **evolução do aprendizado**, treinamos o PPO de novo salvando *checkpoints* a cada 50 mil passos. Depois gravamos **o mesmo cenário** (mesma semente) com o agente em 0, 100 mil, 300 mil, 500 mil e 1 milhão de passos, em um único GIF.

A semente é escolhida pelo melhor resultado do modelo final, então o cenário é **escolhido de propósito**. A média de vários episódios está nas seções anteriores.

In [ ]:
from stable_baselines3.common.callbacks import CheckpointCallback
from PIL import Image as PILImage, ImageDraw, ImageFont

CK = f"{BASE}/ckpt"

env_ck = criar_env()
modelo_ck = PPO("MlpPolicy", env_ck, verbose=0, **PPO_PARAMS)
modelo_ck.save(f"{CK}/ppo_ckpt_0_steps")       # agente sem treino

cb = CheckpointCallback(save_freq=50_000, save_path=CK, name_prefix="ppo_ckpt")
modelo_ck.learn(total_timesteps=1_000_000, callback=cb)
modelo_ck.save(f"{CK}/ppo_ckpt_final")

In [ ]:
final = PPO.load(f"{CK}/ppo_ckpt_final")

# Escolhe o cenário (semente) em que o modelo final pousa melhor
notas = dict(recompensas_por_semente(final, n=30))
SEED = max(notas, key=notas.get)
print("Semente escolhida:", SEED, "| recompensa do modelo final:", round(notas[SEED], 1))

try:
    fonte = ImageFont.load_default(size=22)
except TypeError:
    fonte = ImageFont.load_default()


def legenda(frame, linha1, linha2):
    """Coloca o texto em uma faixa acima da imagem (não cobre a nave)."""
    h, w, _ = frame.shape
    img = PILImage.new("RGB", (w, h + 64), (25, 25, 35))
    img.paste(PILImage.fromarray(frame), (0, 64))
    d = ImageDraw.Draw(img)
    d.text((8, 5), linha1, fill=(255, 255, 255), font=fonte)
    d.text((8, 33), linha2, fill=(255, 255, 255), font=fonte)
    return np.array(img)


def episodio_com_frames(modelo, seed, max_passos=700):
    env = gym.make("LunarLander-v3", render_mode="rgb_array")
    obs, _ = env.reset(seed=seed)
    frames, total = [], 0.0
    for _ in range(max_passos):
        frames.append(env.render())
        acao, _ = modelo.predict(obs, deterministic=True)
        obs, r, term, trunc, _ = env.step(int(acao))
        total += r
        if term or trunc:
            break
    frames.append(env.render())
    env.close()
    return frames, total


momentos = [0, 100_000, 300_000, 500_000, 1_000_000]
todos = []
for m in momentos:
    mod = PPO.load(f"{CK}/ppo_ckpt_{m}_steps")
    frames, total = episodio_com_frames(mod, SEED)
    frames += [frames[-1]] * 30                    # pausa no final do episódio
    texto1 = f"Treino: {m:,} passos".replace(",", ".")
    texto2 = f"Recompensa: {total:.0f}"
    todos += [legenda(f, texto1, texto2) for f in frames]
    print(texto1, "->", texto2)

imageio.mimsave(f"{BASE}/treino_progresso.gif", todos[::2], fps=30, loop=0)
imageio.mimsave(f"{BASE}/treino_progresso.mp4", todos[::2], fps=30, macro_block_size=1)
Image(f"{BASE}/treino_progresso.gif")

**Resultado registrado nesta repetição do treino (mesma configuração, semente 42, 100 episódios):** PPO com 300 mil passos = 109,1 ± 99,4 e com 1 milhão de passos = 251,9 ± 40,4. Os valores ficam próximos dos da seção 6 (79,8 ± 106,7 e 245,4 ± 52,0); a diferença vem da aleatoriedade do treino.

**GIF gerado (semente 6, recompensa do episódio em cada momento):** 0 passos = -619; 100 mil = -447; 300 mil = 81; 500 mil = 282; 1 milhão = 301.